# Rejsekort stop time series and locations
Export two linked Parquet files: `stop_timeseries.parquet` with `time_idx`, `stop_id`, `check_ins`, `check_outs`; and `stop_locations.parquet` with `stop_id`, `stop_name`, `latitude`, `longitude`.
Read cleaned Rejsekort data, without filtering to complete journeys or S-trains. Raw events remain lazy; only aggregated bins are collected. Dense output is written one day at a time. Stop IDs are not merged into stations.

In [ ]:
from pathlib import Path
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
from msc_thesis.data.rejsekort.cleaning import clean_path
from msc_thesis.utils.paths import PROCESSED_DIR

YEAR = 2017
INPUT_PATH = clean_path(YEAR)
GTFS_STOPS_PATH = Path('/mnt/raid/data_sortedmob/RKD/gtfs_dk_2021/stops.txt')
OUTPUT_DIR = PROCESSED_DIR / 'rejsekort' / f'stop_timeseries_{YEAR}'
INTERVAL_MINUTES = 20
COUNT_MODE = 'transactions'
CHECK_TYPE_MAP = {'Fi': 'in', 'Tr': 'in', 'Co': 'out'}
CHECK_TYPE_MAP_CONFIRMED = False
PERSON_TYPE_CODES = [1, 2, 4, 5, 7, 8, 9, 10, 18]
DENSE_OUTPUT = True
START_TIME = None
END_TIME = None

## Configuration and assumptions
Run ingestion and cleaning first, or set `INPUT_PATH` to a cleaned dataset. Adjust the year and matching GTFS path together.
The proposed mapping treats `Fi` and `Tr` as check-ins and `Co` as check-outs. Confirm these meanings for your source, add any other codes (for example `Cc`) only after checking their meaning, and set `CHECK_TYPE_MAP_CONFIRMED = True`. Unknown or missing event codes cause an error, not silent exclusion. Transfer check-ins count as check-ins; no intermediate check-out is inferred.
`transactions` counts event rows. `passengers` sums the three raw passenger fields only for the configured person-type codes, following the old notebook. Verify those codes for your source. Passenger values are not filled across trips; missing or negative counts on person slots cause an error, and events with no recorded people contribute zero.
Bins are left-closed, right-open, labelled by their start. Timestamps are local and timezone-naive, as in cleaning: the repeated autumn DST hour cannot be distinguished without additional source information. Choose a positive interval dividing 24 hours.
Dense output includes every used stop and interval between `START_TIME` (inclusive) and `END_TIME` (exclusive). Defaults cover the full event-date span. Zeros mean no recorded events, not proof that the data feed was complete. Do not include known missing coverage as genuine zero demand.

In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Cleaned dataset not found: {INPUT_PATH}')
if not GTFS_STOPS_PATH.is_file():
    raise FileNotFoundError(f'GTFS stops not found: {GTFS_STOPS_PATH}')
if not isinstance(INTERVAL_MINUTES, int) or INTERVAL_MINUTES <= 0 or 1440 % INTERVAL_MINUTES:
    raise ValueError('INTERVAL_MINUTES must be a positive integer dividing 1440')
if COUNT_MODE not in {'transactions', 'passengers'}:
    raise ValueError('COUNT_MODE must be transactions or passengers')
if not CHECK_TYPE_MAP or not set(CHECK_TYPE_MAP.values()) <= {'in', 'out'}:
    raise ValueError('CHECK_TYPE_MAP values must be in or out')

parquet_files = sorted(INPUT_PATH.rglob('*.parquet')) if INPUT_PATH.is_dir() else [INPUT_PATH]
if not parquet_files:
    raise ValueError('No input Parquet files found')
events = pl.scan_parquet(parquet_files)
code_counts = events.group_by('Model').agg(pl.len().alias('events')).collect().sort('Model')
display(code_counts)
unknown_codes = [code for code in code_counts['Model'].to_list() if code not in CHECK_TYPE_MAP]
if unknown_codes:
    raise ValueError(f'Unmapped Model codes: {unknown_codes}. Verify and update CHECK_TYPE_MAP.')
if not CHECK_TYPE_MAP_CONFIRMED:
    raise ValueError('Review the code counts and confirm CHECK_TYPE_MAP before continuing.')

In [ ]:
invalid_events = events.filter(
    pl.col('event_time').is_null() | pl.col('StopPointNr').is_null()
).select(pl.len()).collect().item()
if invalid_events:
    raise ValueError(f'{invalid_events} events lack timestamps or stop IDs')

if COUNT_MODE == 'passengers':
    invalid_slots = [
        pl.col(f'Passagertype{slot}').is_in(PERSON_TYPE_CODES)
        & (pl.col(f'PassagerAntal{slot}').is_null() | (pl.col(f'PassagerAntal{slot}') < 0))
        for slot in (1, 2, 3)
    ]
    invalid_counts = events.filter(pl.any_horizontal(invalid_slots)).select(pl.len()).collect().item()
    if invalid_counts:
        raise ValueError(f'{invalid_counts} events have missing or negative person counts')
    weight = pl.sum_horizontal([
        pl.when(pl.col(f'Passagertype{slot}').is_in(PERSON_TYPE_CODES))
        .then(pl.col(f'PassagerAntal{slot}')).otherwise(0)
        for slot in (1, 2, 3)
    ])
else:
    weight = pl.lit(1, dtype=pl.Int64)

bounds = events.select(
    pl.col('event_time').min().alias('first'),
    pl.col('event_time').max().alias('last'),
).collect().row(0, named=True)
if bounds['first'] is None:
    raise ValueError('The input dataset is empty')
frequency = f'{INTERVAL_MINUTES}min'
start_time = pd.Timestamp(START_TIME) if START_TIME is not None else pd.Timestamp(bounds['first']).normalize()
end_time = pd.Timestamp(END_TIME) if END_TIME is not None else pd.Timestamp(bounds['last']).normalize() + pd.Timedelta(days=1)
if start_time.tzinfo is not None or end_time.tzinfo is not None:
    raise ValueError('Use timezone-naive bounds matching the cleaned timestamps')
if start_time >= end_time or start_time.floor(frequency) != start_time or end_time.floor(frequency) != end_time:
    raise ValueError('Bounds must be increasing and aligned to interval boundaries')

selected_events = events.filter(
    (pl.col('event_time') >= start_time.to_pydatetime())
    & (pl.col('event_time') < end_time.to_pydatetime())
).with_columns(
    pl.col('StopPointNr').cast(pl.String).alias('stop_id'),
    pl.col('event_time').dt.truncate(f'{INTERVAL_MINUTES}m').alias('time_idx'),
    pl.col('Model').replace_strict(CHECK_TYPE_MAP).alias('check_type'),
    weight.cast(pl.Int64).alias('weight'),
)
binned = selected_events.group_by('time_idx', 'stop_id').agg(
    pl.col('weight').filter(pl.col('check_type') == 'in').sum().alias('check_ins'),
    pl.col('weight').filter(pl.col('check_type') == 'out').sum().alias('check_outs'),
).collect().to_pandas().sort_values(['time_idx', 'stop_id']).reset_index(drop=True)
if binned.empty:
    raise ValueError('No events within the requested bounds')
stop_ids = sorted(binned['stop_id'].unique())
print(f'{len(stop_ids)} stops, {len(binned):,} observed bins; count mode: {COUNT_MODE}')
display(binned.head())

## Stop locations
Keep Rejsekort stop IDs as the output keys. For matching, strip whitespace and leading zeros from numeric GTFS IDs, following the old notebook; reject collisions after normalization. Check unmatched IDs and names before interpreting the lookup as complete. Missing coordinates remain null, with no nearest-station substitution.

In [ ]:
gtfs_stops = pd.read_csv(
    GTFS_STOPS_PATH, dtype={'stop_id': 'string'},
    usecols=['stop_id', 'stop_name', 'stop_lat', 'stop_lon'],
)
gtfs_stops['stop_id'] = gtfs_stops['stop_id'].str.strip().str.replace(r'^0+(?=\d)', '', regex=True)
if gtfs_stops['stop_id'].isna().any() or gtfs_stops['stop_id'].eq('').any():
    raise ValueError('GTFS contains missing stop IDs')
if gtfs_stops['stop_id'].duplicated().any():
    raise ValueError('GTFS stop IDs are not unique after normalization; inspect the mapping')
stop_locations = pd.DataFrame({'stop_id': stop_ids}).merge(
    gtfs_stops, on='stop_id', how='left', validate='one_to_one',
).rename(columns={'stop_lat': 'latitude', 'stop_lon': 'longitude'})
for column in ['latitude', 'longitude']:
    stop_locations[column] = pd.to_numeric(stop_locations[column], errors='raise')
invalid_coordinates = (
    (stop_locations['latitude'].notna() & ~stop_locations['latitude'].between(-90, 90))
    | (stop_locations['longitude'].notna() & ~stop_locations['longitude'].between(-180, 180))
)
if invalid_coordinates.any():
    raise ValueError('GTFS contains coordinates outside valid latitude/longitude ranges')
missing_locations = stop_locations.loc[stop_locations[['latitude', 'longitude']].isna().any(axis=1)]
print(f'{len(missing_locations)} of {len(stop_locations)} stops lack complete coordinates')
display(missing_locations if not missing_locations.empty else stop_locations.head())

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
timeseries_path = OUTPUT_DIR / 'stop_timeseries.parquet'
locations_path = OUTPUT_DIR / 'stop_locations.parquet'
temporary_path = OUTPUT_DIR / 'stop_timeseries.partial.parquet'
output_schema = pa.schema([
    ('time_idx', pa.timestamp('ns')), ('stop_id', pa.string()),
    ('check_ins', pa.int64()), ('check_outs', pa.int64()),
])
expected_totals = binned[['check_ins', 'check_outs']].sum()
written_totals = pd.Series({'check_ins': 0, 'check_outs': 0}, dtype='int64')
rows_written = 0
try:
    with pq.ParquetWriter(temporary_path, output_schema, compression='zstd') as writer:
        for date in pd.date_range(start_time.normalize(), end_time, freq='1D', inclusive='left'):
            day_start = max(date, start_time)
            day_end = min(date + pd.Timedelta(days=1), end_time)
            day_bins = binned.loc[(binned['time_idx'] >= day_start) & (binned['time_idx'] < day_end)]
            if DENSE_OUTPUT:
                intervals = pd.date_range(day_start, day_end, freq=frequency, inclusive='left')
                if len(intervals) * len(stop_ids) > 10_000_000:
                    raise ValueError('Daily grid exceeds 10 million rows; use sparse output or a longer interval')
                grid = pd.MultiIndex.from_product([intervals, stop_ids], names=['time_idx', 'stop_id'])
                day_bins = day_bins.set_index(['time_idx', 'stop_id']).reindex(grid, fill_value=0).reset_index()
            day_bins = day_bins.astype({'check_ins': 'int64', 'check_outs': 'int64'})
            writer.write_table(pa.Table.from_pandas(day_bins, schema=output_schema, preserve_index=False))
            rows_written += len(day_bins)
            written_totals += day_bins[['check_ins', 'check_outs']].sum()
    if not written_totals.equals(expected_totals.astype('int64')):
        raise AssertionError('Export changed the aggregated counts')
    temporary_path.replace(timeseries_path)
except BaseException:
    temporary_path.unlink(missing_ok=True)
    raise
stop_locations.to_parquet(locations_path, index=False, compression='zstd')
print(f'Wrote {rows_written:,} time-series rows to {timeseries_path}')
print(f'Wrote {len(stop_locations):,} stop locations to {locations_path}')
print(f'Totals ({COUNT_MODE}): {written_totals.to_dict()}')